# Tiingo over MCP

Tiingo end-of-day data by two paths: the **MCP server**, which is how consumers
reach it, and `navi`'s `TiingoClient` directly, which is what the server calls
underneath.

Worth running both, because the translation between them is the point. Tiingo's
wire format is camelCase — `exchangeCode`, `startDate`, `adjClose`, `divCash`,
`splitFactor` — and navi carries those as pydantic *aliases* so it can parse the
API. The MCP tools don't publish them: they return meida's own response models,
so a price row arrives as `adj_close` / `div_cash` / `split_factor` with an ISO
`date` string, matching every other source on the server.

**Requires the MCP server running** (`python -m mcp_server.server`) and
`TIINGO_API_KEY` in `../navi/.env`.

In [7]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import asyncio
sys.path.append("../../")   # repo root, for `clients`
from clients import TiingoClient
from lib.utils import print_json_vertical

from utils import (
    list_mcp_tools,
    show_tool_schema,
    get_series_info,
    get_price_series,
)

## 1. Discovery — the Tiingo tools

Two of them. `list_mcp_tools()` with no argument lists every tool on the
server (29 across six sources); the prefix narrows it to this one.

In [8]:
_ = await list_mcp_tools('tiingo_')

tiingo_series_info: Fetch metadata for a Tiingo ticker (ETF, mutual fund, or stock), including its available date range.
tiingo_price_series: Return the end-of-day price series (OHLCV plus split/dividend-adjusted prices) for a Tiingo ticker. Provide start_date/end_date as YYYY-MM-DD for a range; omit for the latest day.


### Schemas

`show_tool_schema` prints the **output** schema as well as the input one. That
half is the interesting one here: it is what states a price row comes back
snake_case with an ISO `date`, and it is what a consumer reads instead of
guessing from an example response.

`count` on the price series is how many rows Tiingo sent, not how many were
mapped — so `count != len(prices)` means rows were dropped in translation
rather than never sent.

In [9]:
_ = await show_tool_schema('tiingo_series_info')

tiingo_series_info
  Fetch metadata for a Tiingo ticker (ETF, mutual fund, or stock), including its available date range.

  -- arguments --
  ticker         string   required

  -- returns --
  ticker         string   required
                          Tiingo's symbol for the instrument, e.g. 'AAPL'. Case-insensitive on input.
  name           string   required
                          Full name of the instrument, e.g. 'Apple Inc'.
  exchange_code  string   optional
                          Listing venue, e.g. 'NASDAQ' or 'NYSE ARCA'. Absent for some funds.
  start_date     string   optional
                          First date with end-of-day data, ISO 'YYYY-MM-DD'. None when Tiingo publishes no coverage bound (typically a ticker with no price history).
  end_date       string   optional
                          Most recent date with end-of-day data, ISO 'YYYY-MM-DD'. Lags the current date for delisted or thinly traded instruments.
  description    string   optional
              

In [10]:
_ = await show_tool_schema('tiingo_price_series')

tiingo_price_series
  Return the end-of-day price series (OHLCV plus split/dividend-adjusted prices) for a Tiingo ticker. Provide start_date/end_date as YYYY-MM-DD for a range; omit for the latest day.

  -- arguments --
  ticker         string   required
  start_date     string   optional
  end_date       string   optional
  resample_freq  string   optional

  -- returns --
  ticker         string   required
                          Tiingo symbol the rows belong to, upper-cased.
  count          integer  required
                          Number of price rows in this response. Derived -- Tiingo reports no total, so this never signals truncation, it just saves counting.
  prices         array    optional
                          One row per trading day in the requested window. Empty when the range covers no trading days, or predates the ticker's history.

  TiingoPriceRow:
  date           string   optional
                          Trading day, ISO 'YYYY-MM-DD' (Tiingo's midnight-UT

## 2. Example calls through MCP

The same two requests the direct-client sections below make, but as a consumer
sees them. Compare the field names against the raw payloads in §3.

In [11]:
info = await get_series_info('SPY')
print_json_vertical(info)   # snake_case, ISO date strings — compare with §3

SPY -- SPDR S&P 500 ETF Trust
  exchange: NYSE
  range:    1993-01-29 -> 2026-09-04
{
  "description": "Historical ETF prices for SPDR S&P 500 ETF (SPY). SPDR S&P 500 ETF Trust (the Trust) is an exchange traded fund. The Trust corresponds to the price and yield performance of the S&P 500 Index. The S&P 500 Index is composed of 500 selected stocks and spans over 24 separate industry groups. The Fund's investment sectors include information technology, financials, energy, health care, consumer staples, industrials, consumer discretionary, materials, utilities and telecommunication services.",
  "end_date": "2026-09-04",
  "exchange_code": "NYSE",
  "name": "SPDR S&P 500 ETF Trust",
  "start_date": "1993-01-29",
  "ticker": "SPY"
}


In [12]:
series = await get_price_series('SPY', start_date='2024-01-02', end_date='2024-01-10')
print_json_vertical(series['prices'][0])   # adj_close / div_cash / split_factor

SPY: 7 rows (count=7)
  2024-01-02  close=   472.65  adj_close=    458.8239  vol=123007793
  2024-01-03  close=   468.79  adj_close=    455.0768  vol=103585866
  2024-01-04  close=   467.28  adj_close=    453.6110  vol=84232169
  2024-01-05  close=   467.92  adj_close=    454.2323  vol=85553758
  2024-01-08  close=   474.60  adj_close=    460.7169  vol=74879074
  ... 2 more
{
  "adj_close": 458.8239124554,
  "adj_high": 459.8140751354,
  "adj_low": 456.7270973683,
  "adj_open": 458.3482460699,
  "adj_volume": 123007793.0,
  "close": 472.65,
  "date": "2024-01-02",
  "div_cash": 0.0,
  "high": 473.67,
  "low": 470.49,
  "open": 472.16,
  "split_factor": 1.0,
  "volume": 123007793
}


## 3. Direct client — what the server calls underneath

Everything below bypasses MCP and uses `navi`'s `TiingoClient`. The `_get`
cells show Tiingo's raw payload; the typed cells show navi's models, which
parse the camelCase via aliases but keep `date`/`datetime` objects rather than
the ISO strings the MCP tools publish.

### Verify credentials

In [13]:
from environment import get_tiingo_api_key, get_tiingo_base_url

key = get_tiingo_api_key()
print(f"Loaded Tiingo key ({len(key)} chars)")
print(f"Tiingo base URL ({get_tiingo_base_url()})")

Loaded Tiingo key (40 chars)
Tiingo base URL (https://api.tiingo.com/tiingo)


### `/daily/<ticker>` — ticker metadata

The raw payload below is where `exchangeCode` / `startDate` / `endDate` come
from — the names §2 does not publish.

In [14]:
async def show_raw_meta(ticker="SPY"):
    async with TiingoClient() as client:
        raw = await client._get(f"/daily/{ticker}")
    print_json_vertical(raw)

await show_raw_meta()

{
  "description": "Historical ETF prices for SPDR S&P 500 ETF (SPY). SPDR S&P 500 ETF Trust (the Trust) is an exchange traded fund. The Trust corresponds to the price and yield performance of the S&P 500 Index. The S&P 500 Index is composed of 500 selected stocks and spans over 24 separate industry groups. The Fund's investment sectors include information technology, financials, energy, health care, consumer staples, industrials, consumer discretionary, materials, utilities and telecommunication services.",
  "endDate": "2026-09-04",
  "exchangeCode": "NYSE",
  "name": "SPDR S&P 500 ETF Trust",
  "startDate": "1993-01-29",
  "ticker": "SPY"
}


In [15]:
async def show_meta(ticker="SPY"):
    async with TiingoClient() as client:
        meta = await client.get_meta(ticker)
    print(f"{meta.ticker} -- {meta.name}")
    print(f"exchange: {meta.exchange_code}")
    print(f"range:    {meta.start_date} -> {meta.end_date}")

await show_meta()

SPY -- SPDR S&P 500 ETF Trust
exchange: NYSE
range:    1993-01-29 -> 2026-09-04


### `/daily/<ticker>/prices` — EOD price series

Provide `startDate`/`endDate` (YYYY-MM-DD) for a range. Works for ETFs, mutual
funds (e.g. `VFIAX`), and stocks. Note the raw bar's `adjClose`, `adjVolume`,
`divCash` and `splitFactor`.

In [16]:
async def show_raw_prices(ticker="SPY", start_date="2024-01-02", end_date="2024-01-10"):
    async with TiingoClient() as client:
        raw = await client._get(
            f"/daily/{ticker}/prices",
            {"startDate": start_date, "endDate": end_date},
        )
    print(f"{len(raw)} rows; first bar:")
    print_json_vertical(raw[0])

await show_raw_prices()

7 rows; first bar:
{
  "adjClose": 458.8239124554,
  "adjHigh": 459.8140751354,
  "adjLow": 456.7270973683,
  "adjOpen": 458.3482460699,
  "adjVolume": 123007793,
  "close": 472.65,
  "date": "2024-01-02T00:00:00.000Z",
  "divCash": 0.0,
  "high": 473.67,
  "low": 470.49,
  "open": 472.16,
  "splitFactor": 1.0,
  "volume": 123007793
}


In [17]:
async def show_prices(ticker="SPY", start_date="2024-01-02", end_date="2024-01-10"):
    async with TiingoClient() as client:
        series = await client.get_prices(ticker, start_date=start_date, end_date=end_date)
    print(f"{series.ticker}: {len(series.prices)} bars")
    print()
    for p in series.prices:
        print(f"{p.date.date()}  close={p.close:>8.2f}  adj_close={p.adj_close:>12.4f}  vol={p.volume}")

await show_prices()

SPY: 7 bars

2024-01-02  close=  472.65  adj_close=    458.8239  vol=123007793
2024-01-03  close=  468.79  adj_close=    455.0768  vol=103585866
2024-01-04  close=  467.28  adj_close=    453.6110  vol=84232169
2024-01-05  close=  467.92  adj_close=    454.2323  vol=85553758
2024-01-08  close=  474.60  adj_close=    460.7169  vol=74879074
2024-01-09  close=  473.88  adj_close=    460.0179  vol=65931439
2024-01-10  close=  476.56  adj_close=    462.6195  vol=67310640


### Full available history

In [18]:
async def show_full_history(ticker="SPY"):
    async with TiingoClient() as client:
        series = await client.get_prices(ticker, start_date="1900-01-01")
    first, last = series.prices[0], series.prices[-1]
    print(f"{series.ticker}: {len(series.prices)} bars "
          f"({first.date.date()} -> {last.date.date()})")

await show_full_history()

SPY: 8458 bars (1993-01-29 -> 2026-09-04)
